In [ ]:
import extraccion.minioFunctions as mf
import pandas as pd
anios = [2022, 2023, 2024, 2025]
dfgen = []
dfs = []
for k in anios:
    dfs2 = mf.bajar_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Suelo2/Suelo2_{k}.parquet', type= 'df')
    dfg = mf.bajar_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Final/final_{k}.parquet', type= 'df' )
    dfs.append(dfs2)    
    dfgen.append(dfg)

final = pd.concat(dfgen, ignore_index=True)
suelo2 = pd.concat(dfs, ignore_index=True)

In [ ]:
l = []
for k in [0,1]:
    print(f"Analizando el año {k}:")
    dataf = mf.bajar_fichero(mf.crear_cliente(), path_server=f"grupo3/raw/Nuevas_Zonas/rusia{k}.parquet", type="df")
    l.append(dataf)

final = pd.concat(l, ignore_index=True)

In [ ]:
# nulos = final[final['soil_temp'].isna()].reset_index()[['lat', 'lon', 'date']]
nulos = final[final['cloud_cover'].isna()].reset_index()[['lat', 'lon', 'date']]
# mf.subir_fichero(mf.crear_cliente(), path_server="grupo3/raw/Incendios_y_no_incendios/suelo_temporal.parquet", df=nulos)
nulos

In [26]:
final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2704 entries, 0 to 2703
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   lat                 2704 non-null   float64       
 1   lon                 2704 non-null   float64       
 2   date                2704 non-null   datetime64[us]
 3   final               2704 non-null   int64         
 4   temp_mean           2699 non-null   float64       
 5   temp_max            2699 non-null   float64       
 6   temp_min            2699 non-null   float64       
 7   humidity_mean       2699 non-null   float64       
 8   precipitation       2699 non-null   float64       
 9   wind_speed_max      2699 non-null   float64       
 10  wind_gusts_max      2699 non-null   float64       
 11  pressure_mean       2699 non-null   float64       
 12  cloud_cover         2699 non-null   float64       
 13  radiation           2699 non-null   float64       
 14  eva

In [ ]:
suelo2.info()

In [ ]:
aniadir = mf.bajar_fichero(mf.crear_cliente(), path_server='grupo3/raw/Suelo2/sin_nulos_rusia.parquet',type='df')
aniadir.sort_values(by='date')

In [ ]:
aniadir = aniadir.set_index(['lat', 'lon'])
final = final.set_index(['lat', 'lon'])
final['soil_temp'] = final['soil_temp'].fillna(aniadir['soil_temp'])

In [24]:
# final = final.reset_index()
final = final.drop(columns=['level_0', 'index'])

In [25]:
mf.subir_fichero(mf.crear_cliente(), path_server="grupo3/raw/Nuevas_Zonas/rusia.parquet", df=final)

Fichero subido como grupo3/raw/Nuevas_Zonas/rusia.parquet


In [ ]:
df_merged = pd.merge(suelo2, final, on=['lat', 'lon'], how='right', indicator=True)
# print(len(df_merged))
df_diferencia = df_merged[df_merged['_merge'] != 'both'].copy()
df_diferencia = df_diferencia.drop(columns=['fire_index', '_merge', 'date_x'])

In [ ]:
suelo2_unico = suelo2.drop_duplicates(subset=['lat', 'lon'], keep='first')
df_merged = pd.merge(final, suelo2_unico, on=['lat', 'lon'], how='left')

df_merged = df_merged.set_index(['lat', 'lon'])
aniadir_idx = aniadir.set_index(['lat', 'lon'])

df_merged['soil_temp'] = df_merged['soil_temp'].fillna(aniadir_idx['soil_temp'])

df_final = df_merged.reset_index()

In [ ]:
df_final = df_final.rename(columns={'date_x':'date'}).sort_values(by='date').reset_index().drop(columns=['date_y', 'index', 'fire_index'])

In [ ]:
df_final.info()

In [ ]:
dfs_por_anio = {
    int(anio): grupo.copy() 
    for anio, grupo in df_final.dropna(subset=['date']).groupby(df_final['date'].dt.year)
}

In [ ]:
for anio, df_anio in dfs_por_anio.items():
    mf.subir_fichero(mf.crear_cliente(), path_server=f'grupo3/raw/Final/final_{anio}.parquet', df=df_anio)

In [ ]:
'''
RENOMBRAR COLUMNAS PARA COMPATIBILIDAD
df = mf.bajar_fichero(mf.crear_cliente(), path_server='grupo3/raw/Incendios_y_no_incendios/suelo_temporal.parquet', type='df')
df = df.rename(columns={'lat_mean':'lat', 'lon_mean':'lon'})
mf.subir_fichero(mf.crear_cliente(), path_server='grupo3/raw/Incendios_y_no_incendios/suelo_temporal.parquet', df=df)
'''